In [ ]:
import requests

url = "http://localhost/api/variants/configs/fetch"
key = ""

headers = {
    "Content-Type": "application/json",
    "Authorization": key,
}

payload = {
    "environment_ref": {
        "slug": "development",
        "version": None,
        "id": None,
    },
    "application_ref": {
        "slug": "CharacterCreation",
        "version": None,
        "id": None,
    },
}

response = requests.post(
    url,
    headers=headers,
    json=payload,   # <-- requests handles JSON encoding
    allow_redirects=True,  # equivalent to -L
)

response.raise_for_status()  # raises if HTTP 4xx/5xx

In [2]:
prompt_info = response.json()["params"]["prompt"]

In [3]:
prompt_info["input_keys"]

['name',
 'age',
 'identity_and_role',
 'origin_story_summary',
 'personality_traits_and_type',
 'emotional_profile',
 'core_values',
 'romances_and_loves',
 'intimacy_profile',
 'detailed_phys_description',
 'voice_and_body_language',
 'sexual_physical_description',
 'sensitivities',
 'soft_kinks_and_limits']

In [4]:
print(prompt_info["messages"][0]["content"])

You are a creative assistant helping build structured character data for an adult-oriented visual novel. 
Given a natural-language biography template filled with character information and the JSON schema below, generate a complete, valid JSON object that matches the schema.

The biography may include personality, emotions, beliefs, upbringing, romantic tendencies, intimacy comfort levels, kinks, limits, physical descriptions, and notable life experiences. 
Interpret the text faithfully, inferring reasonable values when details are implied but not explicitly stated.

**Character Biography Template:**
{{name}} is {{age}} years old.
{{identity_and_role}}  
{{origin_story_summary}} 
{{personality_traits_and_type}} 
{{emotional_profile}} 
{{core_values}} 
{{romances_and_loves}} 
{{intimacy_profile}} 
{{detailed_phys_description}} 
{{voice_and_body_language}} 
{{sexual_physical_description}} 
{{sensitivities}} 
{{soft_kinks_and_limits}} 

**JSON Schema:**
```json
{
"name": "",
"age": 0,
"gen

In [6]:
file_path = 'prologue.md'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        markdown_text = f.read()
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


In [5]:
file_path = 'prologue_summary.md'

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        summary_markdown_text = f.read()
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Here is the refined summary:
Standing by Elowen Creek, Lunessa reflects on her idellic childhood, steeped in comforting rituals. Her mornings begin in her yellow-walled bedroom, moving seamlessly from family breakfast to the bakery where she works side-by-side with her parents. She navigates a village that feels like an extension of herself—visiting the elderly Widow Blackwood to read poetry, gardening with her lifelong best friend Tia, and spending evenings playing cards with her brother Jaxon or sharing lemonade with neighbors. These familiar rhythms are the foundation of her world.
At the center of this life is Lunessa’s naive, romantic heart. Her only romance, a summer relationship with a boy named Liam, was defined by pure innocence—stolen kisses and hand-holding. Though his letters have tapered off, the experience cemented her belief in love as something soft and untainted. In her mind, intimacy belongs to another world, far removed from the safety of Elowen Creek.
Yet, despite t

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
    check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
    max_bucket_size=10,  # Controls the maximum burst size.
)


# Initialize the model with OpenRouter's base URL
model_name = "z-ai/glm-4.5-air:free"
model = init_chat_model(
    model=model_name,
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key="YOUR_OPENROUTER_KEY",
    rate_limiter=rate_limiter
).with_retry(stop_after_attempt=3)



from langchain_classic.chains.summarize import load_summarize_chain

from langchain_core.prompts import PromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=5000, chunk_overlap=100)
splits = text_splitter.split_text(markdown_text)

final_documents=text_splitter.create_documents(splits) 

# Define prompts
chunks_prompt = """
Please summarize the below speech:
Speech: `{text}`
Summary:
"""

map_prompt_template = PromptTemplate(input_variables=['text'], template=chunks_prompt)

final_prompt = """
This text is inteded to be a prologue for a mature NSFW narrative and a character background. Make it a coherent body, reducing 
redundancies. As a rule: You can re-sort the text order, you can use an alias for the protagonist (Lune for Lunessa).

Make it engaging and interesting.
Speech: {text}
"""

final_prompt_template = PromptTemplate(input_variables=['text'], template=final_prompt)


chain = load_summarize_chain(
    llm=model,
    chain_type="refine",
    verbose=True
)

output_summary = chain.run(final_documents)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"*Lunessa stands at the edge of Elowen Creek, her boots sinking slightly into the damp earth. The creek burbles gently behind her, a sound as familiar as her own heartbeat. She gazes across the water towards the rolling hills in the distance, their verdant slopes dotted with cottages that look like dollhouses from here. The air is heavy with the scent of dew and distant woodsmoke, a smell she's known all her life.*

*A memory surfaces unbidden - herself as a child, perched on the banks of this very creek, fishing rod in hand and a mischievous grin plastered across her face. Her father stood nearby, tall and silent, but his presence was comforting. The sun had been warm that day, the sky an endless blue, and she'd felt utterly content.*

*That contentment has followed Lunessa through the years like a faithful dog. Even as she grew older a